# E1.2 — Can geometry alone rescue GPT-2's space? (no similarity training)

E1.1 found the predicted split: mapping DINOv2 into **GPT-2 mean-pooled**
retained 74% of the held-out R² but only 27% of R@1. The mechanism was
measured before the fit — GPT-2's caption space has effective rank **6.5
of 768** and a mean inter-caption cosine of **+0.999**, against DINOv2's
380.2 and +0.031.

The conclusion drawn there was "usable retrieval geometry needs similarity
training". That conclusion is **stronger than the evidence supports**,
because it conflates two things:

| question | E1.1's answer |
|---|---|
| Is the correspondence present in raw LM representations? | **yes** — R² 0.377, shuffle gap 0.505, R@1 at 95x chance |
| Is it usable *raw*? | **no** — R@1 0.095 |
| Is it usable after **unsupervised geometric correction**? | **not tested** |

This notebook tests the third. All four corrections are **purely
geometric and label-free** — no contrastive objective, no pairs, no
training signal of any kind. They only reshape the space.

| arm | correction |
|---|---|
| raw | none — the E1.1 baseline |
| centered | subtract the training mean |
| all-but-top-k | remove the top k principal components (Mu & Viswanath) |
| whitened | rescale so the covariance becomes the identity |

**Discipline:** every transform is fitted on **training rows only** and
applied unchanged to the held-out rows. Fitting on the eval set would
leak, and in a space this anisotropic the leak would be large.

**Validated on synthetic controls first.** On a constructed space matching
GPT-2's regime (pair-cosine +1.000, effective rank 11.3), raw retrieval
scores R@1 = 0.375 and centering lifts it to 1.000 — so the corrections
*can* rescue a space of this shape. Whether they rescue GPT-2's is the
measurement.

**Pre-registered reading:**

| outcome | conclusion |
|---|---|
| a correction recovers most of the bge-m3 arm's R@1 | the correspondence is usable without similarity training; only isotropy is required |
| partial recovery | anisotropy explains part of the gap, contrastive training contributes the rest |
| no recovery | the deficit is in the representation itself, not its coordinate frame — E1.1's original conclusion stands |

In [ ]:
# =====================================================================
# WHERE THIS RUNS AND WHERE DATA GOES - read before running
# =====================================================================
# RUNTIME (choose in the Colab menu, or just open this locally):
#   Colab  : Runtime -> Change runtime type -> T4 / L4 GPU. A free T4 is
#            enough for most notebooks here; CPU works but is slow.
#   Local  : open in Jupyter on a machine with a CUDA GPU or Apple
#            Silicon. Nothing needs changing - the google.colab import
#            below fails harmlessly and it falls through to local mode.
#            To drive a local kernel from the Colab UI: install
#            jupyter_http_over_ws, launch jupyter with
#            --NotebookApp.allow_origin='https://colab.research.google.com'
#            and paste the printed localhost URL (with its token) into
#            Connect -> Connect to a local runtime.
#
# STORAGE (set STORAGE below):
#   "drive" : Google Drive at MyDrive/convergence_experiment  [DEFAULT]
#             Survives session timeouts, so checkpoints resume. On a
#             local machine this falls back to LOCAL_DIR automatically.
#   "local" : LOCAL_DIR on whatever machine is running. On a hosted
#             Colab VM this disk is ERASED at session end - downloads
#             and checkpoints do not survive.
#   "env"   : whatever DATA_DIR is already set to in the environment.
#
# The same folder can be shared between Colab and a local machine (the
# caches are plain .npz) - point both at one synced Drive folder.
# =====================================================================
import os
from pathlib import Path

STORAGE   = "drive"                     # "drive" | "local" | "env"
DRIVE_DIR = "/content/drive/MyDrive/convergence_experiment"
LOCAL_DIR = "./convergence_data"

try:
    import google.colab                 # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if STORAGE == "env":
    assert os.environ.get("DATA_DIR"), "STORAGE='env' but DATA_DIR is unset"
elif STORAGE == "drive" and IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    os.environ["DATA_DIR"] = DRIVE_DIR
else:
    if STORAGE == "drive":
        print("not on Colab - Drive unavailable, using LOCAL_DIR instead")
    os.environ["DATA_DIR"] = str(Path(LOCAL_DIR).resolve())

DATA_DIR = Path(os.environ["DATA_DIR"])
DATA_DIR.mkdir(parents=True, exist_ok=True)

try:
    import torch
    _dev = ("cuda" if torch.cuda.is_available()
            else "mps" if getattr(torch.backends, "mps", None)
            and torch.backends.mps.is_available() else "cpu")
    _name = torch.cuda.get_device_name(0) if _dev == "cuda" else _dev
except ImportError:
    _dev, _name = "?", "torch not installed yet - run the pip cell"
print(f"environment : {'Colab' if IN_COLAB else 'local machine'}")
print(f"device      : {_dev} ({_name})")
print(f"DATA_DIR    : {DATA_DIR}")
if _dev == "cpu":
    print("WARNING: no GPU detected - encoding will take hours, not "
          "minutes")

In [ ]:
import re, os
import numpy as np
from pathlib import Path

# ONE source of truth: name the pairs file, and DERIVE the encoder size
# from it. Setting the size separately is exactly how a large run got
# scored against the base reference arm.
PAIRS_FILE = "crossmodal_pairs_gpt2_dinov2-large_cls+patch.npz"

_m = re.search(r"dinov2-(small|base|large)", PAIRS_FILE)
assert _m, f"cannot read the encoder size from {PAIRS_FILE!r}"
ENCODER_SIZE = _m.group(1)

# bge-m3 reference arms, cls+patch pooling, ~11 rows per input dim
E1_REF = {
    "small": dict(r2=0.516, r1=0.357, r5=0.698, r10=0.826),
    "base":  dict(r2=0.580, r1=0.466, r5=0.809, r10=0.907),
    "large": dict(r2=0.609, r1=0.504, r5=0.835, r10=0.935),
}
BGE = E1_REF[ENCODER_SIZE]

DATA_DIR = Path(os.environ["DATA_DIR"])
rng = np.random.default_rng(0)
ALPHA_GRID = [1e-3, 1e-2, 1e-1, 1.0]
N_EVAL = 1000

print(f"pairs file  : {PAIRS_FILE}")
print(f"encoder size: {ENCODER_SIZE} (derived from the filename)")
print(f"reference   : bge-m3 arm R@1 {BGE['r1']}, R2 {BGE['r2']}")

d = np.load(str(DATA_DIR / PAIRS_FILE))
IMG, TXT = d["img"].astype(np.float64), d["txt"].astype(np.float64)
print(f"IMG {IMG.shape}  TXT {TXT.shape}")

idx = rng.permutation(len(IMG))
te, tr = idx[:N_EVAL], idx[N_EVAL:]
print(f"{len(tr)} train / {len(te)} eval  "
      f"({len(tr)/IMG.shape[1]:.1f} rows per input dim)")

def l2n(V):
    return V / (np.linalg.norm(V, axis=-1, keepdims=True) + 1e-12)

def ridge(X, Y, a):
    return np.linalg.solve(X.T @ X + a * np.eye(X.shape[1]), X.T @ Y)

def recall(S):
    o = np.argsort(-S, axis=1)
    r = (o == np.arange(len(S))[:, None]).argmax(1)
    return {k: float((r < k).mean()) for k in (1, 5, 10)}

def paircos(V):
    Vn = l2n(V)
    i, j = rng.integers(0, len(V), 3000), rng.integers(0, len(V), 3000)
    m = i != j
    return float((Vn[i[m]] * Vn[j[m]]).sum(1).mean())

def effrank(V):
    Vc = V - V.mean(0)
    s = np.linalg.svd(Vc, full_matrices=False, compute_uv=False)
    p = s ** 2 / (s ** 2).sum()
    p = p[p > 0]
    return float(np.exp(-(p * np.log(p)).sum()))

## The four corrections — fitted on train rows, applied to eval rows

In [ ]:
def make_raw(F):
    return lambda A: A

def make_center(F):
    mu = F.mean(0)
    return lambda A: A - mu

def make_abtt(F, k=3):
    mu = F.mean(0)
    _, _, Vt = np.linalg.svd(F - mu, full_matrices=False)
    P = Vt[:k]
    return lambda A: (A - mu) - ((A - mu) @ P.T) @ P

def make_whiten(F):
    mu = F.mean(0)
    C = np.cov((F - mu).T) + 1e-6 * np.eye(F.shape[1])
    ev, V = np.linalg.eigh(C)
    ev = np.clip(ev, 1e-8, None)
    Wm = V @ np.diag(ev ** -0.5) @ V.T
    return lambda A: (A - mu) @ Wm

ARMS = {
    "raw":            make_raw,
    "centered":       make_center,
    "all-but-top-1":  lambda F: make_abtt(F, 1),
    "all-but-top-3":  lambda F: make_abtt(F, 3),
    "all-but-top-10": lambda F: make_abtt(F, 10),
    "whitened":       make_whiten,
}
print("corrections are label-free: they use only TXT[tr], never IMG,")
print("never the pairing, and never the eval rows")

## Run every arm on the identical split

In [ ]:
print(f"{'arm':16s} {'pair-cos':>9s} {'eff.rank':>9s} {'R2':>7s} "
      f"{'R@1':>7s} {'R@5':>7s} {'R@10':>7s} {'% of bge R@1':>13s}")
results = {}
for name, maker in ARMS.items():
    f = maker(TXT[tr])                     # fitted on TRAIN targets only
    Ytr, Yte = f(TXT[tr]), f(TXT[te])

    best = (None, -9)
    nv = max(200, len(tr) // 5)
    for a in ALPHA_GRID:
        Wv = ridge(IMG[tr][:-nv], Ytr[:-nv], a)
        Pv = IMG[tr][-nv:] @ Wv
        sc = 1 - ((Ytr[-nv:] - Pv) ** 2).sum() / \
                 ((Ytr[-nv:] - Ytr[-nv:].mean(0)) ** 2).sum()
        if sc > best[1]:
            best = (a, sc)

    W = ridge(IMG[tr], Ytr, best[0])
    P = IMG[te] @ W
    r2 = 1 - ((Yte - P) ** 2).sum() / ((Yte - Yte.mean(0)) ** 2).sum()
    r = recall(l2n(P) @ l2n(Yte).T)
    results[name] = dict(r2=r2, **{f"r{k}": r[k] for k in (1, 5, 10)},
                         pc=paircos(Yte), er=effrank(Yte), alpha=best[0])
    print(f"{name:16s} {results[name]['pc']:+9.3f} "
          f"{results[name]['er']:9.1f} {r2:7.3f} "
          f"{r[1]:7.3f} {r[5]:7.3f} {r[10]:7.3f} "
          f"{100*r[1]/BGE['r1']:12.1f}%")

print(f"\n{'bge-m3 arm (E1)':16s} {'':>9s} {'':>9s} {BGE['r2']:7.3f} "
      f"{BGE['r1']:7.3f} {BGE['r5']:7.3f} {BGE['r10']:7.3f} "
      f"{100.0:12.1f}%")

## Verdict

In [ ]:
best_arm = max((k for k in results if k != "raw"),
               key=lambda k: results[k]["r1"])
b = results[best_arm]; raw = results["raw"]
lift = b["r1"] - raw["r1"]
share = b["r1"] / BGE["r1"]

print(f"best correction : {best_arm}")
print(f"  R@1 {raw['r1']:.3f} (raw) -> {b['r1']:.3f}   lift {lift:+.3f}")
print(f"  pair-cosine {raw['pc']:+.3f} -> {b['pc']:+.3f}")
print(f"  effective rank {raw['er']:.1f} -> {b['er']:.1f}")
print(f"  reaches {100*share:.1f}% of the bge-m3 arm's R@1\n")

if share > 0.75:
    print("VERDICT: RECOVERED - an unsupervised geometric correction")
    print("  restores most of the retrieval performance. Similarity")
    print("  TRAINING is not required; ISOTROPY is. The correspondence")
    print("  was present in raw LM representations all along and only")
    print("  the coordinate frame hid it.")
elif share > 0.45:
    print("VERDICT: PARTIAL - anisotropy explains a substantial part of")
    print("  the gap but not all of it. Report both the lift and the")
    print("  residual: contrastive training contributes something beyond")
    print("  reshaping the space.")
else:
    print("VERDICT: NOT RECOVERED - reshaping the space does not make the")
    print("  correspondence usable. The deficit is in the representation")
    print("  itself, not merely its coordinate frame, and E1.1's original")
    print("  conclusion stands.")

## Why this matters beyond the project

Cross-model alignment is routinely measured with cosine- and kernel-based
metrics. E1.1 showed those metrics can be near-degenerate in a space at
pair-cosine +0.999 — cosine to target read **1.000** there while retrieval
ran at 9.5%. If a label-free correction moves retrieval substantially,
then part of what such metrics report as *absence of convergence* is
really *absence of isotropy*, and the two are separable with three lines
of linear algebra.

That is a caution applicable to the Platonic Representation Hypothesis
literature itself, and it is independent of whether the hypothesis is
true: measure the geometry of both spaces before interpreting any
similarity number computed in them.